# Tutorial 2: SQD-GQE on real IBM Quantum hardware (train + inference)

This notebook shows how real IBM Quantum hardware plugs into the exact same GQE loop as notebook 1 -- the only thing that changes is the sampling step (`--sampler-kind ibm_hardware` instead of the default `ffsim`). Circuit generation, the loss (GRPO), and the parameter update are completely untouched. See `docs/2026-08-21/pipeline_overview_2026-08-21.pdf` for the full architecture picture and real end-to-end results (32-qubit training to +4.58 mHa, chemical accuracy at 1M-dim inference; 46-qubit zero-shot with dynamical decoupling + GPU recovery to +203 mHa).

**Requires real IBM Quantum credentials and consumes real device time/quota** -- the hardware-executing cells below are clearly marked and will raise a clear error if credentials are missing rather than run silently against nothing.

## 1. Credentials

Copy `.env.example` (repo root) to `.env.local`, fill in your real `IBM_API_KEY` and `IBM_CRN`, then load it into the shell **before** starting Jupyter (env vars set inside the notebook process alone are fine too, e.g. via `os.environ` below, but `.env.local` is the convention used by every training script in this project):

```bash
set -a; source ../.env.local; set +a
jupyter notebook
```

`sqd_gqe/factory.py`'s `IbmHardwareSampler` construction reads `IBM_API_KEY` (token) and `IBM_CRN` (instance) directly from the environment -- see that file for the exact call.

In [1]:
import os

HAVE_IBM_CREDENTIALS = bool(os.environ.get("IBM_API_KEY")) and bool(os.environ.get("IBM_CRN"))
print("IBM credentials present:", HAVE_IBM_CREDENTIALS)
if not HAVE_IBM_CREDENTIALS:
    print("Hardware-executing cells below will be skipped -- set IBM_API_KEY/IBM_CRN to run them for real.")

IBM credentials present: False
Hardware-executing cells below will be skipped -- set IBM_API_KEY/IBM_CRN to run them for real.


## 2. Pick a real backend

`--ibm-backend-name` names the actual IBM Quantum device to sample on (e.g. `ibm_marrakesh`, `ibm_kingston` -- both used for real results in this project). `--backend` (separate flag) selects the *offline* `HardwareBackend` registry entry used for depth/2q-gate cost estimates during training (its coupling map should match the real device you're sampling on, or be a reasonable proxy -- see `sqd_gqe/hw/backends.py`).

`--ibm-dynamical-decoupling` is **not on by default** -- turning it on was a real, measured accuracy improvement in this project (cut a 46-qubit zero-shot error roughly in half before any SQD recovery). There is no reason to leave it off for a real run.

## 3. Training on real hardware

Same script as notebook 1 (`benchmark/amortized_train.py`), one flag changed. All 10 sampled circuits for one training iteration are batched into a **single** Qiskit Runtime job (one queue wait per iteration, not per circuit) -- see `IbmHardwareSampler._sample_batch`'s docstring.

Real runs in this project used `--recover-iterations 10` (GPU-based SQD recovery inside the reward, requires `--diag-backend sbd` and a built native binary -- see `tutorial/03_environment_setup.md`) and 20-30 iterations at ~19-27 minutes each. The cell below intentionally uses a tiny, fast configuration (2 iterations, no recovery) just to demonstrate the real hardware path runs -- **not** a setting you'd train a real result with.

In [2]:
import subprocess, sys

if HAVE_IBM_CREDENTIALS:
    result = subprocess.run(
        [
            sys.executable, "../benchmark/amortized_train.py",
            "--molecules", "../molecules/small/h2.yaml",
            "--backend", "ibm_kingston",
            "--sampler-kind", "ibm_hardware",
            "--ibm-backend-name", "ibm_marrakesh",
            "--ibm-dynamical-decoupling",
            "--iters", "1",
            "--ngates", "4",
            "--shots", "2000",
            "--max-dim", "500",
            "--out", "/tmp/tutorial_h2_hw_run.json",
            "--ckpt", "/tmp/tutorial_h2_hw_policy.pt",
            "--checkpoint-every", "100",
        ],
        capture_output=True, text=True, cwd=".",
    )
    print(result.stdout[-2500:])
    if result.returncode != 0:
        print(result.stderr[-2500:])
else:
    print("Skipped -- no IBM credentials in this environment.")

Skipped -- no IBM credentials in this environment.


## 4. Zero-shot inference on real hardware, with SQD recovery

`sqd_gqe.run.run_molecule` again, this time with `sampler_kind="ibm_hardware"` and `recover=True`. `--recover` re-diagonalizes every sampled circuit's real hardware output through the self-consistent SQD recovery loop (GPU, if `diag_backend="sbd"` is available -- falls back to the CPU path otherwise, which is impractically slow above ~30 qubits, see the architecture deck's wall-time frame).

`recover_subspace_dim` sets the alpha-determinant count per recovery step (net CI dimension = this value squared for a closed-shell/RHF molecule) -- independent of `max_dim`, which only bounds the one-shot (no-recovery) path.

In [3]:
from sqd_gqe.run import run_molecule

if HAVE_IBM_CREDENTIALS:
    report = run_molecule(
        "../molecules/small/h2.yaml",
        backend="ibm_kingston",
        sampler_kind="ibm_hardware",
        ibm_backend_name="ibm_marrakesh",
        ibm_dynamical_decoupling=True,
        checkpoint="/tmp/tutorial_h2_hw_policy_best.pt",
        adapt_iters=1,
        recover=True,
        recover_max_iterations=5,
        recover_subspace_dim=50,
    )
    print(report.summary())
else:
    print("Skipped -- no IBM credentials in this environment.")

Skipped -- no IBM credentials in this environment.


## 5. What to expect at larger qubit counts

Real results from this project, for context (not reproduced by the tiny H2 demo above):

| qubits | stage | error vs. reference | notes |
|---|---|---|---|
| 32 | training, real hardware, full loop | +4.58 mHa | 30 iterations, `--recover-iterations 10` |
| 32 | zero-shot inference, 1M-dim recovery | **+0.0046 mHa** (chemical accuracy) | same checkpoint, larger subspace only |
| 46 | zero-shot, no mitigation | +4,569 mHa | raw `SamplerV2`, no DD, no recovery |
| 46 | zero-shot, + dynamical decoupling | +1,870 mHa | DD alone, still no recovery |
| 46 | zero-shot, + DD + GPU recovery | +203 mHa | 15-step recovery, 1M-dim, ~2.5 min on GPU |

The 46-qubit numbers used a checkpoint trained at 24 qubits (zero-shot transfer, no retraining). See `docs/2026-08-21/pipeline_overview_2026-08-21.pdf` for the full story, including why that residual +203 mHa gap is a generalization problem, not a hardware-noise problem.

## Next steps
- **Environment / multi-GPU setup** (needed for `--diag-backend sbd` recovery at real qubit counts): `tutorial/03_environment_setup.md`.
- **Back to the simulator**: `tutorial/01_simulator_ffsim.ipynb`.
- Real hardware runs are metered, queued resources -- always smoke-test with a tiny `--iters`/`--shots` configuration (as above) before submitting a real multi-hour job.